In [2]:
# Import necessary libraries
import numpy as np
import pandas as pd 
import yfinance as yf
import matplotlib.pyplot as plt 
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, LSTM, Dropout  
from tensorflow.keras.callbacks import EarlyStopping  # Importing EarlyStopping to prevent overfitting by halting training when loss stops improving
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score 
from datetime import datetime, timedelta

def load_stock_data(ticker='GOOGL'):
    end_date = datetime.now()  #
    start_date = end_date - timedelta(days=5*365 + 90)  # Setted start date as 5 years and 3 months before the end date
    stock_data = yf.download(ticker, start=start_date, end=end_date)
    stock_data = stock_data[['Close', 'Open', 'High', 'Low', 'Volume']]  # Select only required columns for analysis
    return stock_data


data = load_stock_data('GOOGL') 

data['SMA_30'] = data['Close'].rolling(window=30).mean()  # Calculate 30-day Simple Moving Average (SMA) of closing prices
data['SMA_100'] = data['Close'].rolling(window=100).mean()  # Calculate 100-day SMA of closing prices
data['Daily_Return'] = data['Close'].pct_change()  # Calculate daily return as percentage change in closing prices

# Remove rows with NaN values that resulted from SMA calculations or daily return calculation
data = data.dropna()  

# Visualize the data with SMA
plt.figure(figsize=(12, 6))  
plt.plot(data['Close'], label='Close Price')  # Plot closing price as the main stock price line
plt.plot(data['SMA_30'], label='30-Day SMA', linestyle='--')
plt.plot(data['SMA_100'], label='100-Day SMA', linestyle='-.') 
plt.title(f'{data.columns[0]} Stock Price with Moving Averages') 
plt.xlabel('Date')
plt.ylabel('Close Price USD ($)')
plt.legend()
plt.grid(True) 
plt.show()  

# Prepare data for scaling
features = ['Close', 'SMA_30', 'SMA_100', 'Daily_Return']  # Define the features to be used for scaling and prediction
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data[features])  # Fit and transform selected features for normalized data

# Define training and testing data (5 years for training, last 1 year for testing)
train_size = int(len(scaled_data) * ((5*365) / (5*365 + 365))) 
train_data = scaled_data[:train_size] 
test_data = scaled_data[train_size - 60:]  # Including previous 60 values in test data

def create_dataset(data, sequence_length=60):
    x, y = [], []  
    for i in range(sequence_length, len(data)): 
        x.append(data[i-sequence_length:i])  
        y.append(data[i, 0])
    return np.array(x), np.array(y) 

x_train, y_train = create_dataset(train_data)  # Creat training sequence and labels
x_test, y_test = create_dataset(test_data)

x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], x_train.shape[2]))  
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], x_test.shape[2]))

# Built the LSTM model with adjusted layers and units to improve R2 score
model = Sequential([  
    LSTM(units=150, return_sequences=True, input_shape=(x_train.shape[1], x_train.shape[2])),  # First LSTM layer with 150 units and outputing sequences
    Dropout(0.3),  # to prevent overfittin
    LSTM(units=150, return_sequences=False),
    Dropout(0.3),
    Dense(units=75),  
    Dense(units=1) 
])

model.compile(optimizer='adam', loss='mean_squared_error') 

early_stop = EarlyStopping(monitor='loss', patience=5)  # Stop training if loss does not improve over 5 epochs

model.fit(x_train, y_train, batch_size=32, epochs=150, callbacks=[early_stop])  # Train model on training data with batch size 32 and up to 150 epochs

# Make predictions on the test data
predictions = model.predict(x_test)  
predictions = scaler.inverse_transform(np.concatenate((predictions, test_data[60:, 1:]), axis=1))[:, 0]  # Scale back predictions to original value

# Visualize actual vs. predicted prices
train = data[:train_size]  # Define the train data portion for visualization
valid = data[train_size:]  # Define the valid (test) data portionn
valid['Predictions'] = predictions  

plt.figure(figsize=(12, 6))  # Set plot size for visualization
plt.plot(train['Close'], label='Train Close Price')  # Plot training data's Close price
plt.plot(valid['Close'], label='Valid Close Price')  # Plot actual Close price of test set
plt.plot(valid['Predictions'], label='Predicted Price', linestyle='--')  # Plot predicted Close price of test set
plt.title(f'{data.columns[0]} Stock Price Prediction') 
plt.xlabel('Date')  
plt.ylabel('Close Price USD ($)') 
plt.legend() 
plt.grid(True)
plt.show() 

# accuracy metrics
mae = mean_absolute_error(valid['Close'], valid['Predictions']) 
mse = mean_squared_error(valid['Close'], valid['Predictions']) 
mape = mean_absolute_percentage_error(valid['Close'], valid['Predictions'])  
r2 = r2_score(valid['Close'], valid['Predictions'])  

print(f"Mean Absolute Error (MAE): {mae}") 
print(f"Mean Squared Error (MSE): {mse}") 
print(f"Mean Absolute Percentage Error (MAPE): {mape}") 
print(f"R² Score: {r2}")

# Predict the next 30 days
last_60_days = test_data[-60:]  # Get the last 60 days of the test data as the initial input for future predictions
future_predictions = []  # Initialize list to store future predictions

for _ in range(30):  # Predict next 30 days
    pred_input = last_60_days[-60:].reshape(1, -1, last_60_days.shape[1]) 
    prediction = model.predict(pred_input) 
    future_predictions.append(prediction[0, 0]) 
    new_entry = np.concatenate((prediction, last_60_days[-1, 1:].reshape(1, -1)), axis=1)  
    # Create new entry with prediction and other features
    last_60_days = np.append(last_60_days, new_entry, axis=0)[1:]  

# Prepare future predictions for inverse scaling
dummy_features = np.zeros((len(future_predictions), scaled_data.shape[1] - 1)) 
scaled_future_predictions = np.concatenate((np.array(future_predictions).reshape(-1, 1), dummy_features), axis=1) 
future_predictions = scaler.inverse_transform(scaled_future_predictions)[:, 0]  # Inverse scale predictions

#  next 30 days
future_dates = [valid.index[-1] + timedelta(days=i) for i in range(1, 31)]  # Generate dates for 30-day predictions


# Visualize the future predictions with ensemble
plt.figure(figsize=(14, 8))
# Plot historical Close price
plt.plot(data['Close'], label='Close Price History', color='blue')  
# Plot LSTM test predictions
plt.plot(valid.index, valid['Predictions'], label='LSTM Predictions', color='green', linestyle='--') 
# Plot future LSTM predictions
plt.plot(future_dates, future_predictions, label='LSTM 30-Day Prediction', color='orange', linestyle='--') 
#plt.plot(future_dates, ensemble_prediction, label='Ensemble Prediction', color='purple', linestyle='-.')  # Plot
plt.title(f'{data.columns[0]} Stock Price Prediction with Ensemble (Next 30 Days)') 
plt.xlabel('Date') 
plt.ylabel('Close Price USD ($)')  
plt.legend() 
plt.grid(True)
plt.show() 

#MAKING MAGNIFIED FORM
from matplotlib.dates import DateFormatter  # Import DateFormatter to format date labels on x-axis

# Set the date range to the last 5 months
five_months_ago = data.index[-1] - pd.DateOffset(months=5) 
train = train[train.index >= five_months_ago]
valid = valid[valid.index >= five_months_ago]

# Plot the graph with magnified view of the last 5 months
plt.figure(figsize=(12, 6)) 

# Plot last 5 months of the train data (actual stock prices)
plt.plot(train['Close'], label='Train Close Price', color='blue')  
plt.plot(valid['Close'], label='Valid Close Price', color='orange')  
plt.plot(valid['Predictions'], label='Predicted Price', linestyle='--', color='green')  
plt.title(f'{data.columns[0]} Stock Price Prediction (Last 5 Months)')  
plt.xlabel('Date')  
plt.ylabel('Close Price USD ($)')  

# Display the date labels on x-axis
plt.gca().xaxis.set_major_formatter(DateFormatter('%Y-%m-%d')) 
plt.xticks(rotation=45)  # Rotate date labels for better readability

plt.legend()  
plt.grid(True) 
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'sklearn'

In [3]:
# Save LSTM artifacts for Streamlit integration
import os
import pickle

ARTIFACT_DIR = "artifacts/lstm"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save the trained Keras model separately (recommended format for TF/Keras models)
model.save(os.path.join(ARTIFACT_DIR, "lstm_model.keras"))

# Save preprocessing + runtime metadata in pickle files
with open(os.path.join(ARTIFACT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(ARTIFACT_DIR, "features.pkl"), "wb") as f:
    pickle.dump(features, f)

with open(os.path.join(ARTIFACT_DIR, "last_sequence.pkl"), "wb") as f:
    pickle.dump(last_60_days, f)

metrics = {
    "mae": float(mae),
    "mse": float(mse),
    "mape": float(mape),
    "r2": float(r2)
}
with open(os.path.join(ARTIFACT_DIR, "metrics.pkl"), "wb") as f:
    pickle.dump(metrics, f)

history_payload = {
    "close_series": data["Close"],
    "valid_close": valid["Close"],
    "valid_predictions": valid["Predictions"]
}
with open(os.path.join(ARTIFACT_DIR, "history_payload.pkl"), "wb") as f:
    pickle.dump(history_payload, f)

future_payload = {
    "future_dates": future_dates,
    "future_predictions": future_predictions.tolist() if hasattr(future_predictions, "tolist") else list(future_predictions)
}
with open(os.path.join(ARTIFACT_DIR, "future_payload.pkl"), "wb") as f:
    pickle.dump(future_payload, f)

print(f"Saved LSTM artifacts in: {ARTIFACT_DIR}")
print("Files:")
for name in sorted(os.listdir(ARTIFACT_DIR)):
    print("-", name)

NameError: name 'model' is not defined